In [8]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets

In [9]:
# device config 
device =  torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# defining the hyperparameters.
EPOCHS = 30
BATCH_SIZE = 100
LEARNING_RATE = 0.001

# defining the Image preprocessing stuffs 
im_transform = transforms.Compose([
    transforms.Pad(4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32),
    transforms.ToTensor()
])

# downloading the CIFAR-10 dataset. 
train_dataset = datasets.CIFAR10('./data', train=True, transform=im_transform, download=True)
test_dataset = datasets.CIFAR10('./data', train=False, transform=transforms.ToTensor())

# loading the datasets. 
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [10]:
def conv3x3(in_channels, out_channels, stride=1): 
    return nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, stride=stride, padding=1, bias=False)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, down_sampling= None) -> None:         
        super(ResidualBlock, self).__init__()

        self.conv1 = conv3x3(in_channels=in_channels, out_channels=out_channels, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = conv3x3(in_channels=out_channels, out_channels=out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.down_sampling = down_sampling

    def forward(self, x):

        residual = x.clone()

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.down_sampling:
            residual = self.down_sampling(x)

        out += residual

        out = self.relu(out)

        return out
        

In [11]:
class ResNet(nn.Module):
    def __init__(self, res_block, num_classes=10) -> None:
        super(ResNet, self).__init__()

        self.conv = conv3x3(in_channels=3, out_channels=16)
        self.bn = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)
        self.res_block = res_block

        self.residual_layer1 = self.make_layer(16, 16)
        self.residual_layer2 = self.make_layer(16, 32)
        self.residual_layer3 = self.make_layer(32, 64)

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)


    def make_layer(self, in_channels, out_channels):
        down_sampling = None
        stride = 1

        if in_channels != out_channels:
            down_sampling = nn.Sequential(conv3x3(in_channels=in_channels, out_channels=out_channels, stride=2), nn.BatchNorm2d(out_channels))
            stride = 2

        res_blocks = []
        res_blocks.append(self.res_block(in_channels=in_channels, out_channels=out_channels, stride=stride, down_sampling= down_sampling))

        res_blocks.append(self.res_block(in_channels=out_channels, out_channels=out_channels))

        return nn.Sequential(*res_blocks)
    

    def forward(self, x):

        out = self.conv(x)
        out = self.bn(out)
        out = self.relu(out)

        out = self.residual_layer1(out)
        out = self.residual_layer2(out)
        out = self.residual_layer3(out)

        out = self.avgpool(out)
        out = torch.flatten(out, 1)        
        out = self.fc(out)

        return out

In [12]:
model = ResNet(res_block=ResidualBlock)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [13]:
# training ts

decay = 0
model.train()

for epoch in range(1, EPOCHS+1): 

    if (epoch + 1)%20 == 0:
        decay+=1
        optimizer.param_groups[0]['lr'] *= 0.5**decay

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1)%100 == 0:
                print(f"Epoch {epoch}/{EPOCHS}, Loss: {loss.item()}")


Epoch 1/30, Loss: 1.5993212461471558
Epoch 1/30, Loss: 1.4482312202453613
Epoch 1/30, Loss: 1.4198590517044067
Epoch 1/30, Loss: 1.0767046213150024
Epoch 1/30, Loss: 1.0106338262557983
Epoch 2/30, Loss: 1.0575991868972778
Epoch 2/30, Loss: 0.9069758653640747
Epoch 2/30, Loss: 1.0076736211776733
Epoch 2/30, Loss: 0.7623130083084106
Epoch 2/30, Loss: 0.8953617811203003
Epoch 3/30, Loss: 1.0003540515899658
Epoch 3/30, Loss: 0.7776042819023132
Epoch 3/30, Loss: 0.902778148651123
Epoch 3/30, Loss: 0.8955861926078796
Epoch 3/30, Loss: 0.7315739393234253
Epoch 4/30, Loss: 0.8205435276031494
Epoch 4/30, Loss: 0.5696231722831726
Epoch 4/30, Loss: 0.9182166457176208
Epoch 4/30, Loss: 0.9734388589859009
Epoch 4/30, Loss: 0.578278660774231
Epoch 5/30, Loss: 0.7076777815818787
Epoch 5/30, Loss: 0.7782566547393799
Epoch 5/30, Loss: 0.5505399703979492
Epoch 5/30, Loss: 0.733878493309021
Epoch 5/30, Loss: 0.7099671959877014
Epoch 6/30, Loss: 0.5319421887397766
Epoch 6/30, Loss: 0.6046561598777771
Epoc

In [14]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        predicted = predicted.to(device)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Test Accuracy: {100*correct/total} ")

Test Accuracy: 86.13 
